In [ ]:
from echospec.figures import FigureVariant, apply_figure_style, save_figure

VARIANT = FigureVariant.PAPER
apply_figure_style(VARIANT)


In [1]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import qutip_jax as qj
from diffrax import PIDController, Tsit5
from jax import default_device, devices, grad, jacfwd, jacrev, jit
from qutip import (CoreOptions, about, basis, destroy, lindblad_dissipator,
                   liouvillian, mcsolve, mesolve, projection, qeye, settings,
                   sigmam, sigmax, sigmay, sigmaz, spost, spre, sprepost,
                   steadystate, tensor)

%matplotlib inline

In [2]:
# system parameters
ed = 1
GammaL = 1
GammaR = 1

# simulation parameters
options = {
    "method": "diffrax",
    "normalize_output": False,
    "stepsize_controller": PIDController(rtol=1e-7, atol=1e-7),
    "solver": Tsit5(scan_kind="bounded"),
    "progress_bar": False,
}

In [3]:
gamma = 0.1  # dissipation rate

In [4]:
# time dependent drive
@jit
def driving_coeff(t, omega):
    return jnp.cos(omega * t)


# system Hamiltonian
def setup_system():
    H_0 = sigmaz()
    H_1 = sigmax()
    H = [H_0, [H_1, driving_coeff]]
    return H

In [12]:
import numpy as np

# simulation parameters
psi0 = basis(2, 0)
tlist = np.linspace(0.0, 10.0, 100)
c_ops = [np.sqrt(gamma) * sigmam()]
e_ops = [projection(2, 1, 1)]

In [13]:
# Objective function: returns final exc. state population
def f(omega):
    H = setup_system()
    arg = {"omega": omega}
    result = mcsolve(H, psi0, tlist, c_ops, e_ops=e_ops, ntraj=100, args=arg)
    return result.expect[0][-1]


In [14]:
grad_f = grad(f)(2.0)

JaxRuntimeError: UNKNOWN: -:0:0: error: unknown attribute code: 22
-:0:0: note: in bytecode version 6 produced by: StableHLO_v1.13.0
